# Attention maps exploration

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display


def plot_attention_maps(
    analyzer,
    source,                         # DatasetPromptSource o StreamingDatasetPromptSource
    prompt_indices: list[int],      # es. [0, 1, 2]
    layers: list[int] | None = None,  # None = tutti i layer
    heads:  list[int] | None = None,  # None = tutte le heads
    figsize_per_cell: tuple = (2.0, 2.0),
    cmap: str = "Blues",
    save: bool = False,
    out_dir: str = ".",
):
    """
    Per ogni prompt_idx in prompt_indices, plotta una griglia
    n_layers × n_heads delle attention maps (post-softmax).
    Ogni cella è la heatmap (seq_len × seq_len) di una (layer, head).
    """
    for prompt_idx in prompt_indices:

        # ── 1. Estrai il prompt ───────────────────────────────────────────────
        prompt_ids = source.get_prompt(prompt_idx)           # torch.Tensor (seq_len,)
        input_ids  = prompt_ids.unsqueeze(0).to(analyzer.device)
        seq_len    = input_ids.shape[1]

        # Decodifica token per le label sull'asse
        tokens = analyzer.tokenizer.convert_ids_to_tokens(
            input_ids[0].cpu().tolist()
        )
        # Label compatte: tronca a 6 char per non sporcare gli assi
        tick_labels = [t[:6].replace("▁", "_") for t in tokens]

        # ── 2. Forward pass con output_attentions=True ────────────────────────
        with torch.no_grad():
            outputs = analyzer.model(
                input_ids,
                output_attentions=True,
                use_cache=False,
            )

        # outputs.attentions: tuple di (n_layers,) tensori shape (1, n_heads, seq, seq)
        all_attentions = outputs.attentions   # tuple[Tensor]
        n_layers_model = len(all_attentions)
        n_heads_model  = all_attentions[0].shape[1]

        # ── 3. Selezione layer e heads ────────────────────────────────────────
        sel_layers = layers if layers is not None else list(range(n_layers_model))
        sel_heads  = heads  if heads  is not None else list(range(n_heads_model))

        n_l = len(sel_layers)
        n_h = len(sel_heads)

        # ── 4. Costruzione griglia ────────────────────────────────────────────
        fig, axes = plt.subplots(
            n_l, n_h,
            figsize=(figsize_per_cell[0] * n_h, figsize_per_cell[1] * n_l),
            constrained_layout=True,
        )
        axes = np.atleast_2d(axes)

        MAX_TICKS = 16   # mostra al massimo 16 tick per asse per leggibilità

        for ri, layer_idx in enumerate(sel_layers):
            attn_layer = all_attentions[layer_idx][0]   # (n_heads, seq, seq)
            attn_np    = attn_layer.float().cpu().numpy()

            for ci, head_idx in enumerate(sel_heads):
                ax  = axes[ri, ci]
                mat = attn_np[head_idx]   # (seq, seq)

                im = ax.imshow(mat, cmap=cmap, aspect="auto",
                               vmin=0, vmax=mat.max(),
                               interpolation="nearest")

                # Tick label solo se seq_len non è troppo grande
                if seq_len <= MAX_TICKS:
                    ax.set_xticks(range(seq_len))
                    ax.set_yticks(range(seq_len))
                    ax.set_xticklabels(tick_labels, fontsize=5, rotation=90)
                    ax.set_yticklabels(tick_labels, fontsize=5)
                else:
                    # Mostra solo ogni k-esimo tick
                    step = max(1, seq_len // MAX_TICKS)
                    ax.xaxis.set_major_locator(ticker.MultipleLocator(step))
                    ax.yaxis.set_major_locator(ticker.MultipleLocator(step))
                    ax.tick_params(labelsize=5)

                # Header riga (layer) e colonna (head)
                if ci == 0:
                    ax.set_ylabel(f"L{layer_idx}", fontsize=7,
                                  rotation=0, labelpad=18, va="center")
                if ri == 0:
                    ax.set_title(f"H{head_idx}", fontsize=7, pad=3)

        # ── 5. Colorbar globale ───────────────────────────────────────────────
        # Usiamo l'ultimo imshow come riferimento
        cbar = fig.colorbar(im, ax=axes, shrink=0.4, pad=0.005, aspect=40)
        cbar.ax.tick_params(labelsize=6)
        cbar.set_label("Attention weight", fontsize=7)

        # ── 6. Titolo e salvataggio ───────────────────────────────────────────
        src_name = getattr(source, "dataset_name", "source")
        fig.suptitle(
            f"Attention Maps — Prompt #{prompt_idx}  |  {src_name}  |  seq_len={seq_len}",
            fontsize=10, fontweight="bold"
        )

        if save:
            path = f"{out_dir}/attn_prompt{prompt_idx}_{'_'.join(map(str,sel_layers))}.png"
            fig.savefig(path, dpi=150, bbox_inches="tight")
            print(f"Saved: {path}")

        display(fig)
        plt.close(fig)

        # Libera memoria GPU
        del outputs, all_attentions
        torch.cuda.empty_cache()

In [ ]:
from core.analyzer import LightweightAttentionAnalyzer
from config import MODEL_NAME

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
#MODEL_NAME = "Qwen/Qwen3-4B"

analyzer = LightweightAttentionAnalyzer(
    MODEL_NAME,
    device="mps",
)
print("Model Succesfully Loaded")

[Analyzer] Loading model 'mistralai/Mistral-7B-Instruct-v0.3' on device 'mps'...


Fetching 3 files:   0%|          | 0/3 [00:15<?, ?it/s]


In [ ]:
from data.prompt_sources import DatasetPromptSource, RandomTokenPromptSource, StreamingDatasetPromptSource
from config import DATASET_NAME, DATASET_CONFIG, DATASET_SPLIT, DATASET_TEXT_COLUMN, MIN_CHARS

wiki_source = DatasetPromptSource(
    tokenizer=analyzer.tokenizer,
    dataset_name=DATASET_NAME,
    dataset_config=DATASET_CONFIG,
    target_tokens=64,
    split=DATASET_SPLIT,
    text_column=DATASET_TEXT_COLUMN,
    min_chars=MIN_CHARS,
)

print("Sources OK")

In [ ]:
# Tutti i layer e heads — pesante su seq_len=512, meglio selezionare
'''
plot_attention_maps(
    analyzer,
    source         = wiki_source,
    prompt_indices = [0, 1, 2],
    layers         = [0, 8, 16, 24, 31],   # 5 layer rappresentativi
    heads          = list(range(0, 32, 4)), # ogni 4 heads
    cmap           = "Blues",
    save           = False,
)
'''
# Solo un layer specifico — tutti gli heads
plot_attention_maps(
    analyzer,
    source         = wiki_source,
    prompt_indices = [0],
    layers         = [20],
    heads          = None,   # tutte le 32 heads
    figsize_per_cell = (1.5, 1.5),
)